# 利用直接 ANN 預測模型之重抽樣分析與 Bootstrap 推論

## 314657010

## 一、總攬說明

這份 notebook 是原本 `HW3_314657010` 的 ANN benchmark 版本。和原作業不同的地方在於：

- 不再先做 Kalman filter decomposition
- 不再拆成 low-frequency 與 high-frequency 兩部分
- 直接對原始觀測時間序列建立 ANN forecasting model
- 仍然保留 Monte Carlo、rolling out-of-sample residual bootstrap、confidence interval、bias / variance / MSE、以及 bootstrap hypothesis testing

這樣可以把它當成 hybrid model 的 benchmark，檢查「直接 ANN」和「Kalman filter + ARIMA + ANN hybrid」之間的差異。

## 二、研究動機

原本作業的重點是研究 hybrid forecasting model 的預測表現與統計推論性質。不過，如果要進一步做比較研究，一個很自然的 benchmark 就是：不做訊號分解，直接對原始時間序列資料建立 ANN forecasting model。

因此，這份延伸 notebook 的目標是：

- 比較 direct ANN forecasting model 在相同模擬架構下的表現
- 保持樣本生成方式與 bootstrap 評估方式不變
- 觀察 direct ANN model 的 bias、variance、MSE、coverage、interval width 與 bootstrap hypothesis testing 結果

## 三、模型設定與設計

這裡使用和原 HW3 相同的模擬資料生成方式，觀測值仍由低頻線性成分、高頻非線性成分與 measurement noise 所組成。不過在模型設計上，本 notebook 不進行 Kalman filter decomposition，而是直接使用原始觀測序列作為 ANN 的輸入。

整體流程如下：

1. 生成模擬時間序列資料
2. 使用前段訓練樣本建立 direct ANN model
3. 以 recursive forecasting 方式產生未來預測值
4. 在訓練區間內使用 rolling out-of-sample residuals 建立 bootstrap residual pool
5. 透過 bootstrap residual resampling 建構 percentile interval 與 normal interval
6. 利用 Monte Carlo simulation 評估 bias、variance、MSE 與 hypothesis testing 結果

In [1]:
import matplotlib
import numpy as np
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tools.sm_exceptions import ConvergenceWarning

matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)


def simulate_babu_style_data(n_steps, noise_std=0.15):
    low_true = np.zeros(n_steps, dtype=float)
    high_true = np.zeros(n_steps, dtype=float)

    low_true[0] = np.random.normal(0, 1.0)
    high_true[0] = np.random.normal(0, 0.5)
    high_true[1] = np.random.normal(0, 0.5)

    for idx in range(1, n_steps):
        shared_error = np.random.normal(0, 1.0)
        low_true[idx] = 0.6 * low_true[idx - 1] + shared_error

        if idx < 2:
            continue

        coeff_1 = 0.5 + 0.9 * np.exp(-(high_true[idx - 1] ** 2))
        coeff_2 = -0.8 - 1.8 * np.exp(-(high_true[idx - 1] ** 2))
        high_true[idx] = (
            coeff_1 * high_true[idx - 1]
            + coeff_2 * high_true[idx - 2]
            + shared_error
        )

    noise = np.random.normal(0, noise_std, size=n_steps)
    measurements = low_true + high_true + noise
    return low_true, high_true, measurements


def save_forecast_trend_plot(result, output_path, model_label):
    horizon_steps = np.arange(1, len(result["final_forecast"]) + 1)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.fill_between(
        horizon_steps,
        result["interval_lower"],
        result["interval_upper"],
        color="tab:blue",
        alpha=0.20,
        label="interval",
    )
    ax.plot(
        horizon_steps,
        result["final_forecast"],
        color="tab:blue",
        marker="o",
        linewidth=2.0,
        label="Forecast",
    )
    ax.plot(
        horizon_steps,
        result["measurements_test"],
        color="tab:red",
        marker="s",
        linewidth=2.0,
        label="Actual",
    )
    ax.set_title(f"Prediction Interval (h={len(horizon_steps)})")
    ax.set_xlabel("Forecast horizon step")
    ax.set_ylabel("Value")
    ax.legend(loc="best")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def create_windows(series, window_size):
    x_data = []
    y_data = []

    for idx in range(window_size, len(series)):
        x_data.append(series[idx - window_size:idx])
        y_data.append(series[idx])

    return np.array(x_data), np.array(y_data)


def fit_direct_ann_bundle(train_series, window_size):
    x_train, y_train = create_windows(train_series, window_size)
    if len(x_train) == 0:
        raise ValueError("Not enough samples for ANN windows. Increase training size or reduce window_size.")

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    x_train_scaled = x_scaler.fit_transform(x_train)
    y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

    ann_model = MLPRegressor(
        hidden_layer_sizes=(10, 10, 10),
        activation="relu",
        solver="adam",
        alpha=1e-5,
        learning_rate_init=0.001,
        max_iter=3000,
        random_state=42
    )
    ann_model.fit(x_train_scaled, y_train_scaled)

    lower_q, upper_q = np.quantile(train_series, [0.01, 0.99])
    iqr = np.subtract(*np.quantile(train_series, [0.75, 0.25]))
    margin = max(0.5 * iqr, 0.25)
    clip_bounds = (float(lower_q - margin), float(upper_q + margin))

    return {
        "model": ann_model,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "clip_bounds": clip_bounds,
        "window_size": window_size,
    }


def recursive_ann_forecast(model, x_scaler, y_scaler, history, horizon, window_size, clip_bounds=None):
    rolling = list(history[-window_size:])
    forecasts = []

    for _ in range(horizon):
        features = np.array(rolling[-window_size:], dtype=float).reshape(1, -1)
        features_scaled = x_scaler.transform(features)
        pred_scaled = model.predict(features_scaled).reshape(-1, 1)
        pred = y_scaler.inverse_transform(pred_scaled)[0, 0]
        if clip_bounds is not None:
            pred = float(np.clip(pred, clip_bounds[0], clip_bounds[1]))
        forecasts.append(pred)
        rolling.append(pred)

    return np.array(forecasts, dtype=float)


def direct_ann_point_forecast(train_series, horizon, window_size=15):
    ann_bundle = fit_direct_ann_bundle(train_series, window_size)
    final_forecast = recursive_ann_forecast(
        ann_bundle["model"],
        ann_bundle["x_scaler"],
        ann_bundle["y_scaler"],
        train_series,
        horizon,
        window_size,
        clip_bounds=ann_bundle["clip_bounds"]
    )
    return {"final_forecast": final_forecast, "ann_bundle": ann_bundle}


def rolling_calibration_residuals(series, initial_train_size, calibration_end, block_size, window_size=15):
    residuals = []

    for start in range(initial_train_size, calibration_end, block_size):
        stop = min(start + block_size, calibration_end)
        horizon = stop - start
        train_series = np.asarray(series[:start], dtype=float)
        actual_block = np.asarray(series[start:stop], dtype=float)

        forecast_bundle = direct_ann_point_forecast(train_series, horizon, window_size=window_size)
        block_residuals = actual_block - forecast_bundle["final_forecast"]
        residuals.extend(block_residuals.tolist())

    return np.asarray(residuals, dtype=float)


def sort_bootstrap_residuals(residual_pool, horizon, rng):
    sampled_indices = rng.choice(len(residual_pool), size=horizon, replace=True)
    sampled_indices.sort()
    return residual_pool[sampled_indices]


def bootstrap_paths_for_interval(point_forecast, calibration_residuals, horizon, bootstrap_replications=300, random_seed=123):
    rng = np.random.default_rng(random_seed)
    bootstrap_paths = np.zeros((bootstrap_replications, horizon), dtype=float)

    for boot_idx in range(bootstrap_replications):
        sampled_residuals = sort_bootstrap_residuals(calibration_residuals, horizon, rng)
        bootstrap_paths[boot_idx, :] = point_forecast + sampled_residuals

    return bootstrap_paths


def percentile_bootstrap_interval(bootstrap_paths, alpha=0.05):
    interval_lower = np.quantile(bootstrap_paths, alpha / 2, axis=0)
    interval_upper = np.quantile(bootstrap_paths, 1 - alpha / 2, axis=0)
    return interval_lower, interval_upper


def normal_bootstrap_interval(bootstrap_paths, z_value=1.959963984540054):
    path_mean = np.mean(bootstrap_paths, axis=0)
    path_std = np.std(bootstrap_paths, axis=0, ddof=1)
    interval_lower = path_mean - z_value * path_std
    interval_upper = path_mean + z_value * path_std
    return interval_lower, interval_upper


def summarize_estimator(forecasts, truths):
    forecasts = np.asarray(forecasts, dtype=float)
    truths = np.asarray(truths, dtype=float)
    errors = forecasts - truths
    bias = np.mean(errors, axis=0)
    variance = np.var(forecasts, axis=0, ddof=1)
    mse = np.mean(errors ** 2, axis=0)

    return {
        "bias_by_horizon": bias,
        "variance_by_horizon": variance,
        "mse_by_horizon": mse,
        "mean_abs_bias": float(np.mean(np.abs(bias))),
        "mean_variance": float(np.mean(variance)),
        "mean_mse": float(np.mean(mse)),
        "h1_bias": float(bias[0]),
        "h1_variance": float(variance[0]),
        "h1_mse": float(mse[0]),
        "errors_h1": errors[:, 0],
    }


def bootstrap_mean_test(sample, null_value=0.0, bootstrap_replications=2000, random_seed=321):
    sample = np.asarray(sample, dtype=float)
    observed_mean = float(np.mean(sample))
    centered_sample = sample - observed_mean + null_value
    rng = np.random.default_rng(random_seed)

    bootstrap_means = np.empty(bootstrap_replications, dtype=float)
    for idx in range(bootstrap_replications):
        draw = centered_sample[rng.choice(len(centered_sample), size=len(centered_sample), replace=True)]
        bootstrap_means[idx] = np.mean(draw)

    p_value = float(np.mean(np.abs(bootstrap_means - null_value) >= abs(observed_mean - null_value)))
    ci_draws = sample[rng.choice(len(sample), size=(bootstrap_replications, len(sample)), replace=True)].mean(axis=1)
    ci_lower, ci_upper = np.quantile(ci_draws, [0.025, 0.975])

    return {
        "observed_mean": observed_mean,
        "p_value": p_value,
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
    }


def run_single_experiment(train_size=100, horizon=1, window_size=15, bootstrap_replications=50, initial_train_size=None, calibration_block=None):
    dt = 1.0
    n_steps = train_size + horizon
    t = np.arange(n_steps) * dt

    low_true, high_true, measurements = simulate_babu_style_data(n_steps)

    if initial_train_size is None:
        initial_train_size = max(window_size + 20, train_size // 2)
    if calibration_block is None:
        calibration_block = max(1, min(10, max(1, train_size - initial_train_size)))

    calibration_residuals = rolling_calibration_residuals(
        measurements,
        initial_train_size=initial_train_size,
        calibration_end=train_size,
        block_size=calibration_block,
        window_size=window_size,
    )

    full_train_bundle = direct_ann_point_forecast(
        measurements[:train_size],
        horizon,
        window_size=window_size,
    )

    final_forecast = full_train_bundle["final_forecast"]
    true_clean_test = low_true[train_size:] + high_true[train_size:]

    bootstrap_paths = bootstrap_paths_for_interval(
        final_forecast,
        calibration_residuals,
        horizon,
        bootstrap_replications=bootstrap_replications,
    )
    interval_lower, interval_upper = percentile_bootstrap_interval(bootstrap_paths, alpha=0.05)
    normal_interval_lower, normal_interval_upper = normal_bootstrap_interval(bootstrap_paths)

    clean_in_interval = (true_clean_test >= interval_lower) & (true_clean_test <= interval_upper)
    clean_in_normal_interval = (true_clean_test >= normal_interval_lower) & (true_clean_test <= normal_interval_upper)

    return {
        "t": t,
        "train_size": train_size,
        "measurements": measurements,
        "measurements_test": measurements[train_size:],
        "true_clean_test": true_clean_test,
        "final_forecast": final_forecast,
        "interval_lower": interval_lower,
        "interval_upper": interval_upper,
        "normal_interval_lower": normal_interval_lower,
        "normal_interval_upper": normal_interval_upper,
        "final_clean_mse": float(np.mean((final_forecast - true_clean_test) ** 2)),
        "clean_coverage": float(np.mean(clean_in_interval) * 100),
        "normal_clean_coverage": float(np.mean(clean_in_normal_interval) * 100),
        "mean_interval_width": float(np.mean(interval_upper - interval_lower)),
        "normal_mean_interval_width": float(np.mean(normal_interval_upper - normal_interval_lower)),
    }


def monte_carlo_summary(experiment_results):
    final_forecasts = np.array([result["final_forecast"] for result in experiment_results], dtype=float)
    clean_truths = np.array([result["true_clean_test"] for result in experiment_results], dtype=float)
    final_stats = summarize_estimator(final_forecasts, clean_truths)
    mean_error_test = bootstrap_mean_test(final_stats["errors_h1"], null_value=0.0)
    return {
        "final": final_stats,
        "bootstrap_mean_error_test": mean_error_test,
    }

## 四、模擬結果

下面直接輸出和原 HW3 相同型態的結果：

- Final forecast estimator summary
- Bootstrap CI comparison
- Bootstrap hypothesis test on horizon-1 forecast error

In [2]:
np.random.seed(42)
n_monte_carlo = 50
train_size = 100
horizons = [20]
bootstrap_replications = 50

print(f"Monte Carlo runs               : {n_monte_carlo}")
print(f"Observed sample size T         : {train_size}")
print(f"Bootstrap replications B       : {bootstrap_replications}")
print("")
print("h | Mean final MSE | Percentile coverage | Normal coverage | Percentile width | Normal width")
print("--|----------------|---------------------|-----------------|------------------|-------------")

for horizon in horizons:
    experiment_results = []
    for run_idx in range(n_monte_carlo):
        np.random.seed(42 + run_idx)
        experiment_results.append(
            run_single_experiment(
                train_size=train_size,
                horizon=horizon,
                bootstrap_replications=bootstrap_replications,
            )
        )

    clean_mses = np.array([result["final_clean_mse"] for result in experiment_results], dtype=float)
    clean_coverages = np.array([result["clean_coverage"] for result in experiment_results], dtype=float)
    normal_clean_coverages = np.array([result["normal_clean_coverage"] for result in experiment_results], dtype=float)
    mean_interval_widths = np.array([result["mean_interval_width"] for result in experiment_results], dtype=float)
    normal_mean_interval_widths = np.array([result["normal_mean_interval_width"] for result in experiment_results], dtype=float)

    estimator_summary = monte_carlo_summary(experiment_results)["final"]
    mean_error_test = monte_carlo_summary(experiment_results)["bootstrap_mean_error_test"]

    print(
        f"{horizon} | "
        f"{np.mean(clean_mses):.4f}         | "
        f"{np.mean(clean_coverages):.2f}%                | "
        f"{np.mean(normal_clean_coverages):.2f}%            | "
        f"{np.mean(mean_interval_widths):.4f}           | "
        f"{np.mean(normal_mean_interval_widths):.4f}"
    )

    print("")
    print("Final forecast estimator summary")
    print(f"Mean absolute bias              : {estimator_summary['mean_abs_bias']:.4f}")
    print(f"Mean variance                  : {estimator_summary['mean_variance']:.4f}")
    print(f"Mean MSE                       : {estimator_summary['mean_mse']:.4f}")
    print(f"Horizon 1 bias                 : {estimator_summary['h1_bias']:.4f}")
    print(f"Horizon 1 variance             : {estimator_summary['h1_variance']:.4f}")
    print(f"Horizon 1 MSE                  : {estimator_summary['h1_mse']:.4f}")
    print("")
    print("Bootstrap CI comparison")
    print(f"Percentile mean width          : {np.mean(mean_interval_widths):.4f}")
    print(f"Normal mean width              : {np.mean(normal_mean_interval_widths):.4f}")
    print(f"Percentile mean clean coverage : {np.mean(clean_coverages):.2f}%")
    print(f"Normal mean clean coverage     : {np.mean(normal_clean_coverages):.2f}%")
    print("")
    print("Bootstrap hypothesis test on horizon-1 forecast error")
    print("H0: E[e1] = 0")
    print(f"Observed mean error            : {mean_error_test['observed_mean']:.4f}")
    print(f"Bootstrap p-value              : {mean_error_test['p_value']:.4f}")
    print(f"bootstrap CI               : [{mean_error_test['ci_lower']:.4f}, {mean_error_test['ci_upper']:.4f}]")
    representative = experiment_results[0]
    plot_path = f"HW3_direct_ann_trend_h{horizon}.png"
    save_forecast_trend_plot(representative, plot_path, model_label="Direct ANN")
    print(f"Saved forecast trend plot      : {plot_path}")
    print("")

Monte Carlo runs               : 50
Observed sample size T         : 100
Bootstrap replications B       : 50

h | Mean final MSE | Percentile coverage | Normal coverage | Percentile width | Normal width
--|----------------|---------------------|-----------------|------------------|-------------
20 | 34.4642         | 80.90%                | 85.00%            | 19.3172           | 21.9363

Final forecast estimator summary
Mean absolute bias              : 0.5235
Mean variance                  : 57.8299
Mean MSE                       : 34.4642
Horizon 1 bias                 : -0.4692
Horizon 1 variance             : 50.4518
Horizon 1 MSE                  : 11.1603

Bootstrap CI comparison
Percentile mean width          : 19.3172
Normal mean width              : 21.9363
Percentile mean clean coverage : 80.90%
Normal mean clean coverage     : 85.00%

Bootstrap hypothesis test on horizon-1 forecast error
H0: E[e1] = 0
Observed mean error            : -0.4692
Bootstrap p-value              :